In [4]:
from datetime import datetime, timezone
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
from random import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Replace these with your InfluxDB token, organization, and bucket
org = "ur3e"
bucket = "ur3e"
token = "-BZITnY_s_YijW3jrGyzuOgiQxxMtXEU9U_yxpBLYlUiPZw0kKvethzTBR2A4NqZuVVw-2bKQ70rDUuPgNCQJQ=="

# Initialize the client
client = InfluxDBClient(url="http://localhost:8086", token=token, org=org)
write_api = client.write_api(write_options=SYNCHRONOUS)   
query_api = client.query_api()

In [ ]:
time_start = "-5m"
time_stop = "now()"
window = "1s"

query_sensor_data = f'''
from(bucket: "ur3e")
  |> range(start: {time_start}, stop: {time_stop})
  |> filter(fn: (r) => r["_measurement"] == "sensor_data")
  |> filter(fn: (r) => r["_field"] =~ /(q|qd)_actual_joint_[0-6]/)
  |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
'''

df_sensor_data = query_api.query_data_frame(query_sensor_data)

In [6]:
print(df_sensor_data.head())

    result  table                            _time  \
0  _result      0 2026-04-21 07:33:00.398708+00:00   
1  _result      0 2026-04-21 07:33:00.447976+00:00   
2  _result      0 2026-04-21 07:33:00.499304+00:00   
3  _result      0 2026-04-21 07:33:00.549832+00:00   
4  _result      0 2026-04-21 07:33:00.598115+00:00   

                            _start                            _stop  \
0 2026-04-21 07:33:00.371931+00:00 2026-04-21 07:38:00.371931+00:00   
1 2026-04-21 07:33:00.371931+00:00 2026-04-21 07:38:00.371931+00:00   
2 2026-04-21 07:33:00.371931+00:00 2026-04-21 07:38:00.371931+00:00   
3 2026-04-21 07:33:00.371931+00:00 2026-04-21 07:38:00.371931+00:00   
4 2026-04-21 07:33:00.371931+00:00 2026-04-21 07:38:00.371931+00:00   

  _measurement     source  q_actual_joint_0  q_actual_joint_1  \
0  sensor_data  pt_mockup          0.000015          0.628321   
1  sensor_data  pt_mockup          0.000015          0.628321   
2  sensor_data  pt_mockup          0.000015          

In [ ]:
q_tags = [f"q_actual_joint_{i}" for i in range(6)]

q_tags.append("_time")

df_pos = df_sensor_data[q_tags].copy()

print(df_pos.head())

In [ ]:
qd_tags = [f"qd_actual_joint_{i}" for i in range(6)]

qd_tags.append("_time")

df_vel = df_sensor_data[qd_tags].copy()

print(df_vel.head())

# Velocity Profile

In [ ]:
df_vel_trunk = df_vel.iloc[300:400]
np_vel = df_vel_trunk.to_numpy()
time_series = df_vel_trunk["_time"]
norm_time = (time_series - time_series.iloc[0]).dt.total_seconds().to_numpy()

for joint in range(6):
    plt.plot(norm_time, np_vel[:, joint], label=f"qd_actual_joint_{joint}")

plt.xlabel("Time (s)")
plt.ylabel("Joint Velocity (rad/s)")
plt.grid()
plt.legend()

# Position Profile

In [ ]:
df_pos_trunk = df_pos.iloc[300:400]
np_pos = df_pos_trunk.to_numpy()
time_series = df_pos_trunk["_time"]
norm_time = (time_series - time_series.iloc[0]).dt.total_seconds().to_numpy()

for joint in range(6):
    plt.plot(norm_time, np_pos[:, joint], label=f"q_actual_joint_{joint}")

plt.xlabel("Time (s)")
plt.ylabel("Joint Position (rad)")
plt.grid()
plt.legend()